# 10. Paoay: summarize land-cover guidance

**One task:** turn notebook 09’s broad-cover probabilities into per-half-month guidance with explicit uncertainty. Everything runs locally; no imagery, training or prediction jobs are launched.

This is predicted guidance, not ground truth. Bare is not verified fallow, and crops does not identify carrot/cabbage/potato. The crop model’s valid inputs remain available regardless of this guidance. See the [workflow guide](../../docs/paoay_workflow.md).


## 1. Setup and explicit rules

Finish notebooks 08–09 first. Starting rules: **at least two observed dates, top probability ≥ 0.60 and margin over the runner-up ≥ 0.10**. Ties remain uncertain. These adjustable thresholds are illustrative, not validated for Paoay or selected using local labels/model scores. Changing them creates a separate result bundle.


In [ ]:
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import land_cover
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

rules = {"minimum_dates": 2, "minimum_top_probability": 0.60, "minimum_margin": 0.10}
show_preview = True
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
dynamic_world_pointer = paoay_root / "dynamic_world/dynamic_world_stage.json"
land_cover_root = paoay_root / "land_cover"
land_cover_pointer = land_cover_root / "land_cover_stage.json"
write_stage_pointer(land_cover_pointer, status="pending", reason="Notebook 10 has not finished the current guidance run")
stages = ["Read verified probability inputs", "Build land-cover guidance", "Review coverage and areas", "Preview evidence states", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook(
    "10_paoay_land_cover", stages, report_folder.parent / "progress", {"rules": rules},
)


## 2. Read verified inputs

Use notebook 09’s successful handoff, including its exact dates/grid and verified Sentinel-2 dependency. Missing, pending or changed inputs stop this run.


In [ ]:
pipeline_progress.begin_stage("Read verified probability inputs")
dynamic_world_handoff = resolve_stage_pointer(dynamic_world_pointer)
request = land_cover.prepare_request(dynamic_world_handoff, land_cover_root, rules)
print("Period:", request["identity"]["period"]["start"], "to", request["identity"]["period"]["end"], "(end exclusive)")
show_table([{"rule": key.replace("_", " "), "value": value} for key, value in rules.items()], title="Illustrative guidance rules")
show_table([{"code": code, "meaning": label} for code, label in land_cover.STATES.items()], title="Guidance states")
pipeline_progress.end_stage()


## 3. Build separate guidance outputs

Each observed pixel keeps its leading class, top/runner-up probabilities, margin and observation count. The guidance state distinguishes missing evidence, too few dates, uncertain probabilities, crop support and other-cover support. Outside the ROI stays nodata.

Rules apply to each half-month independently, allowing changes during rotations. Other-cover support describes that period; it is not permanent agricultural exclusion. No valid crop-model input is removed here.


In [ ]:
pipeline_progress.begin_stage("Build land-cover guidance")
def interval_progress(items):
    return pipeline_progress.track(items, "Summarize land-cover guidance", unit="half-month")
land_cover_handoff = land_cover.build(request, iterator=interval_progress)
bundle = land_cover.load_land_cover_bundle(land_cover_handoff)
pipeline_progress.end_stage()


## 4. Review coverage and areas

Within each half-month, supported class areas plus the three unresolved categories account for every ROI pixel. Areas use 10 m pixel centres; they are predicted cover summaries, not surveyed parcel areas. Do not add repeated half-month areas into one study-area estimate.


In [ ]:
pipeline_progress.begin_stage("Review coverage and areas")
show_table(bundle["coverage"], title="Evidence states by half-month (pixels)")
nonzero_areas = [{**row, "pixels": int(row["pixels"]), "area_ha": float(row["area_ha"]), "roi_percent": float(row["roi_percent"])}
                 for row in bundle["class_areas"] if int(row["pixels"]) > 0]
show_table(nonzero_areas, title="Supported cover and unresolved area by half-month")
print("Full class-area table:", bundle["folder"] / "class_areas.csv")
print("Supported broad cover does not establish local accuracy or crop identity.")
pipeline_progress.end_stage()


## 5. Preview evidence states

A local preview shows where the evidence supports a broad-cover decision and where it remains unresolved. This is not the final combined classification map.


In [ ]:
pipeline_progress.begin_stage("Preview evidence states")
if show_preview:
    from IPython.display import display
    import matplotlib.pyplot as plt
    figure = land_cover.plot_guidance(bundle)
    preview = land_cover_root / "previews" / (bundle["manifest"]["bundle_id"] + ".png")
    preview.parent.mkdir(parents=True, exist_ok=True)
    if not preview.exists():
        figure.savefig(preview, dpi=150)
    display(figure)
    plt.close(figure)
    print("Guidance preview:", preview)
write_stage_pointer(land_cover_pointer, land_cover_handoff)
pipeline_progress.end_stage()


## 6. Summary and next action

Keep crop-only predictions available alongside any eventual combined map. Unknown broad cover must stay unresolved; it is not automatically another crop.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Verified guidance handoff:", land_cover_handoff)
print("Current guidance pointer:", land_cover_pointer)
print("Crop-model inputs: preserved; this is guidance, not a hard exclusion mask.")
print("Next: notebook 11 will calculate the saved crop model's exact features in a separate batch.")
print("Dynamic World attribution:", bundle["manifest"]["attribution"])
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
